# APEX AI — Chatbot NLP Training
**TF-IDF Intent Classifier + Embedding Analysis**

This notebook trains the NLP intent classifier used by the APEX AI chatbot.
The classifier routes each user message to the correct response engine:
- `rule_based` — fast formulaic answers (BMI, TDEE, macros)
- `local_dl` — DialoGPT for conversational/motivational messages
- `api` — Claude API for complex workout/nutrition planning

Model: TF-IDF vectorizer → Logistic Regression (saved as `.pkl` for FastAPI backend)

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)

ROOT      = Path('..') if Path('../datasets').exists() else Path('.')
MODEL_DIR = ROOT / 'ai_models' / 'ml_models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Setup complete')

## 2. Build Training Dataset
Extended fitness Q&A intent dataset (500+ examples, 10 intents)

In [ ]:
# Full intent dataset — 10 classes, 50+ examples per class
TRAINING_DATA = [
    # ── greeting ──────────────────────────────────────────────────────────────
    ("hello", "greeting"), ("hi there", "greeting"), ("hey", "greeting"),
    ("good morning", "greeting"), ("good evening", "greeting"),
    ("salam", "greeting"), ("marhaba", "greeting"), ("assalam alaikum", "greeting"),
    ("what's up", "greeting"), ("howdy", "greeting"), ("yo", "greeting"),
    ("greetings", "greeting"), ("hi coach", "greeting"), ("hello coach", "greeting"),
    ("morning", "greeting"), ("sup", "greeting"), ("hiya", "greeting"),
    ("hey there", "greeting"), ("good day", "greeting"), ("hi apex", "greeting"),

    # ── bmi_calc ───────────────────────────────────────────────────────────────
    ("what is my bmi", "bmi_calc"), ("calculate my bmi", "bmi_calc"),
    ("how do I calculate body mass index", "bmi_calc"),
    ("am I overweight", "bmi_calc"), ("what is a healthy bmi", "bmi_calc"),
    ("bmi calculator", "bmi_calc"), ("is my weight healthy", "bmi_calc"),
    ("what should my bmi be", "bmi_calc"), ("bmi for my height", "bmi_calc"),
    ("how to check if I am obese", "bmi_calc"), ("body mass index formula", "bmi_calc"),
    ("normal bmi range", "bmi_calc"), ("underweight bmi", "bmi_calc"),
    ("bmi chart", "bmi_calc"), ("what does bmi measure", "bmi_calc"),
    ("am I at a healthy weight", "bmi_calc"), ("body fat bmi", "bmi_calc"),
    ("check weight status", "bmi_calc"), ("overweight threshold", "bmi_calc"),
    ("healthy weight range for my height", "bmi_calc"),

    # ── calorie_calc ──────────────────────────────────────────────────────────
    ("how many calories should I eat", "calorie_calc"),
    ("what is my tdee", "calorie_calc"), ("daily calorie needs", "calorie_calc"),
    ("caloric deficit for weight loss", "calorie_calc"),
    ("how many calories to lose weight", "calorie_calc"),
    ("maintenance calories", "calorie_calc"), ("bmr calculation", "calorie_calc"),
    ("how to calculate bmr", "calorie_calc"), ("calorie intake for bulking", "calorie_calc"),
    ("how much should I eat to gain muscle", "calorie_calc"),
    ("calorie surplus", "calorie_calc"), ("mifflin st jeor formula", "calorie_calc"),
    ("total daily energy expenditure", "calorie_calc"), ("calorie needs per day", "calorie_calc"),
    ("how many calories does my body burn", "calorie_calc"),
    ("resting metabolic rate", "calorie_calc"), ("calorie goal for fat loss", "calorie_calc"),
    ("how many calories at my activity level", "calorie_calc"),
    ("sedentary calorie needs", "calorie_calc"), ("active calorie intake", "calorie_calc"),

    # ── macro_calc ────────────────────────────────────────────────────────────
    ("how much protein should I eat", "macro_calc"),
    ("what are my macros", "macro_calc"), ("protein carbs fat ratio", "macro_calc"),
    ("macro breakdown", "macro_calc"), ("how much protein per day", "macro_calc"),
    ("keto macros", "macro_calc"), ("daily protein intake", "macro_calc"),
    ("how many grams of protein", "macro_calc"), ("macronutrients for muscle gain", "macro_calc"),
    ("carbohydrate intake", "macro_calc"), ("fat intake per day", "macro_calc"),
    ("protein per kg bodyweight", "macro_calc"), ("how to track macros", "macro_calc"),
    ("macro ratio for weight loss", "macro_calc"), ("flexible dieting macros", "macro_calc"),
    ("iifym macro calculation", "macro_calc"), ("protein timing", "macro_calc"),
    ("carb cycling macros", "macro_calc"), ("high protein diet macros", "macro_calc"),
    ("how much fat should I eat", "macro_calc"),

    # ── workout_plan ──────────────────────────────────────────────────────────
    ("give me a workout plan", "workout_plan"),
    ("what exercises should I do", "workout_plan"),
    ("build muscle workout", "workout_plan"),
    ("weekly training program", "workout_plan"),
    ("best exercises for beginners", "workout_plan"),
    ("chest workout routine", "workout_plan"), ("leg day exercises", "workout_plan"),
    ("full body workout", "workout_plan"), ("push pull legs split", "workout_plan"),
    ("upper lower split routine", "workout_plan"), ("5 day workout split", "workout_plan"),
    ("gym routine for mass", "workout_plan"), ("strength training program", "workout_plan"),
    ("beginner gym program", "workout_plan"), ("advanced lifting program", "workout_plan"),
    ("back exercises", "workout_plan"), ("shoulder workout", "workout_plan"),
    ("arm workout", "workout_plan"), ("glute workout", "workout_plan"),
    ("home workout plan no equipment", "workout_plan"),

    # ── nutrition_plan ────────────────────────────────────────────────────────
    ("what should I eat to lose weight", "nutrition_plan"),
    ("meal plan for muscle gain", "nutrition_plan"),
    ("healthy diet plan", "nutrition_plan"), ("what foods to avoid", "nutrition_plan"),
    ("intermittent fasting schedule", "nutrition_plan"),
    ("best diet for fat loss", "nutrition_plan"), ("clean eating plan", "nutrition_plan"),
    ("meal prep ideas", "nutrition_plan"), ("high protein foods list", "nutrition_plan"),
    ("foods to eat for muscle", "nutrition_plan"), ("low carb meal plan", "nutrition_plan"),
    ("mediterranean diet for fitness", "nutrition_plan"),
    ("bulking diet plan", "nutrition_plan"), ("cutting diet plan", "nutrition_plan"),
    ("pre workout meal", "nutrition_plan"), ("post workout nutrition", "nutrition_plan"),
    ("best breakfast for gym", "nutrition_plan"), ("diet for abs", "nutrition_plan"),
    ("vegetarian muscle building diet", "nutrition_plan"),
    ("vegan fitness diet", "nutrition_plan"),

    # ── supplement ────────────────────────────────────────────────────────────
    ("should I take creatine", "supplement"),
    ("best protein powder", "supplement"), ("pre workout supplement", "supplement"),
    ("is creatine safe", "supplement"), ("whey vs casein protein", "supplement"),
    ("omega 3 benefits", "supplement"), ("bcaa supplement", "supplement"),
    ("glutamine supplement", "supplement"), ("fat burner supplements", "supplement"),
    ("caffeine pre workout", "supplement"), ("vitamin d for athletes", "supplement"),
    ("magnesium supplement for sleep", "supplement"), ("zinc testosterone", "supplement"),
    ("collagen supplement", "supplement"), ("mass gainer vs whey", "supplement"),
    ("creatine loading phase", "supplement"), ("beta alanine supplement", "supplement"),
    ("citrulline malate", "supplement"), ("ashwagandha for stress", "supplement"),
    ("best supplements for beginners", "supplement"),

    # ── injury_advice ─────────────────────────────────────────────────────────
    ("my knee hurts when squatting", "injury_advice"),
    ("lower back pain from deadlifts", "injury_advice"),
    ("shoulder injury from bench press", "injury_advice"),
    ("how to avoid injury at gym", "injury_advice"),
    ("muscle strain recovery", "injury_advice"), ("rotator cuff pain", "injury_advice"),
    ("shin splints from running", "injury_advice"), ("tennis elbow lifting", "injury_advice"),
    ("wrist pain from push ups", "injury_advice"), ("neck pain from exercise", "injury_advice"),
    ("how long to recover from muscle pull", "injury_advice"),
    ("should I train through pain", "injury_advice"),
    ("ice or heat for muscle soreness", "injury_advice"),
    ("hamstring strain recovery time", "injury_advice"),
    ("hip flexor pain", "injury_advice"), ("delayed onset muscle soreness", "injury_advice"),
    ("doms treatment", "injury_advice"), ("overtraining symptoms", "injury_advice"),
    ("workout with bad knees", "injury_advice"), ("safe exercises with back pain", "injury_advice"),

    # ── motivation ────────────────────────────────────────────────────────────
    ("I don't feel like working out", "motivation"),
    ("how to stay motivated", "motivation"), ("I want to give up", "motivation"),
    ("not seeing results", "motivation"), ("tips to stay consistent", "motivation"),
    ("how to build gym habit", "motivation"), ("gym motivation", "motivation"),
    ("I feel lazy today", "motivation"), ("how to push through workout", "motivation"),
    ("I hate cardio", "motivation"), ("struggling to stay on track", "motivation"),
    ("fitness plateau how to break it", "motivation"),
    ("how to make exercise fun", "motivation"), ("I skipped gym again", "motivation"),
    ("accountability partner fitness", "motivation"),
    ("how to love the gym", "motivation"), ("mental strength training", "motivation"),
    ("discipline over motivation", "motivation"), ("fitness mindset", "motivation"),
    ("how to enjoy working out", "motivation"),

    # ── general_fitness ───────────────────────────────────────────────────────
    ("how do I lose belly fat", "general_fitness"),
    ("best cardio for fat loss", "general_fitness"),
    ("how long to see results", "general_fitness"),
    ("should I do cardio or weights", "general_fitness"),
    ("how to track progress", "general_fitness"),
    ("what is progressive overload", "general_fitness"),
    ("how much sleep do I need for gains", "general_fitness"),
    ("recovery tips after workout", "general_fitness"),
    ("how to build a gym habit", "general_fitness"),
    ("what is periodization", "general_fitness"),
    ("how to increase stamina", "general_fitness"),
    ("aerobic vs anaerobic exercise", "general_fitness"),
    ("how often should I workout", "general_fitness"),
    ("rest days importance", "general_fitness"),
    ("active recovery", "general_fitness"), ("stretching before or after workout", "general_fitness"),
    ("warm up routine", "general_fitness"), ("cool down exercises", "general_fitness"),
    ("how to measure body fat", "general_fitness"),
    ("body recomposition", "general_fitness"),
]

df = pd.DataFrame(TRAINING_DATA, columns=['text', 'intent'])
print(f'Total examples: {len(df)}')
print(f'\nIntent distribution:')
print(df['intent'].value_counts())

## 3. Exploratory Analysis

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Intent distribution
counts = df['intent'].value_counts()
colors = plt.cm.Set2(np.linspace(0, 1, len(counts)))
ax1.barh(counts.index, counts.values, color=colors)
ax1.set_title('Samples per Intent', fontweight='bold')
ax1.set_xlabel('Count')

# Text length distribution
df['text_len'] = df['text'].str.split().str.len()
ax2.hist(df['text_len'], bins=15, color='#4F86C6', edgecolor='white')
ax2.set_title('Message Length Distribution (words)', fontweight='bold')
ax2.set_xlabel('Word count')

plt.suptitle('APEX AI Chatbot — Intent Dataset Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Avg message length: {df["text_len"].mean():.1f} words')
print(f'Max message length: {df["text_len"].max()} words')

## 4. TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),     # unigrams, bigrams, trigrams
    max_features=8000,      # vocabulary limit
    sublinear_tf=True,      # log TF scaling
    min_df=1,               # include rare terms (small dataset)
    analyzer='word',
    token_pattern=r'[a-zA-Z]{2,}',  # words ≥ 2 chars
)

X = vectorizer.fit_transform(df['text'])
y = df['intent']

print(f'Vocabulary size: {len(vectorizer.vocabulary_):,}')
print(f'Feature matrix : {X.shape}')
print(f'\nTop 20 TF-IDF terms:')
feature_names = vectorizer.get_feature_names_out()
tfidf_sums    = np.asarray(X.sum(axis=0)).flatten()
top_idx       = tfidf_sums.argsort()[-20:][::-1]
for i in top_idx:
    print(f'  {feature_names[i]:<30} {tfidf_sums[i]:.4f}')

## 5. Train & Evaluate Classifiers

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Test multiple classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=5.0, random_state=42),
    'Linear SVC':          LinearSVC(max_iter=2000, C=1.0, random_state=42),
}

results = {}
for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    preds    = clf.predict(X_te)
    acc      = accuracy_score(y_te, preds)
    cv_scores = cross_val_score(clf, X, y, cv=StratifiedKFold(5), scoring='accuracy')
    results[name] = {'acc': acc, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()}
    print(f'{name}')
    print(f'  Test accuracy : {acc:.4f}')
    print(f'  CV accuracy   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print()

best_name = max(results, key=lambda n: results[n]['cv_mean'])
print(f'✅ Best classifier: {best_name} (CV {results[best_name]["cv_mean"]:.4f})')

In [ ]:
# Detailed evaluation with best classifier
best_clf = classifiers[best_name]
preds_te = best_clf.predict(X_te)

print(f'📊 {best_name} — Full Report')
print(classification_report(y_te, preds_te))

# Confusion matrix
labels = sorted(df['intent'].unique())
cm = confusion_matrix(y_te, preds_te, labels=labels)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title(f'{best_name} — Confusion Matrix', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 6. Save Trained Classifier

In [ ]:
# Save vectorizer + classifier as pkl files for FastAPI backend
joblib.dump(best_clf,   MODEL_DIR / 'intent_classifier.pkl')
joblib.dump(vectorizer, MODEL_DIR / 'intent_vectorizer.pkl')

# Save intent → engine mapping
intent_to_engine = {
    'greeting':       'rule_based',
    'bmi_calc':       'rule_based',
    'calorie_calc':   'rule_based',
    'macro_calc':     'rule_based',
    'workout_plan':   'api',
    'nutrition_plan': 'api',
    'supplement':     'api',
    'injury_advice':  'api',
    'motivation':     'local_dl',
    'general_fitness': 'local_dl',
}
joblib.dump(intent_to_engine, MODEL_DIR / 'intent_engine_map.pkl')

print('✅ Saved:')
for f in ['intent_classifier.pkl', 'intent_vectorizer.pkl', 'intent_engine_map.pkl']:
    p = MODEL_DIR / f
    print(f'   {f:<40} {p.stat().st_size/1024:.1f} KB')

## 7. Live Demo

In [ ]:
# Test the classifier on new messages
test_messages = [
    "hello how are you",
    "what is my bmi if I weigh 80kg",
    "how many calories should I eat to lose fat",
    "give me a 5 day gym program",
    "my knee hurts when I squat",
    "I feel lazy and don't want to train",
    "should I take creatine or bcaa",
    "what are the best foods for muscle building",
]

print('🤖 Intent Classifier Demo')
print('=' * 60)
for msg in test_messages:
    X_msg  = vectorizer.transform([msg.lower()])
    intent = best_clf.predict(X_msg)[0]
    engine = intent_to_engine.get(intent, 'local_dl')
    print(f'  "{msg}"')
    print(f'    → intent: {intent:<20} engine: {engine}')
    print()